# Dataset

## RealCamVid Dataset

In [ ]:
import torch
from src.options import opt_dict
from src.data.realcamvid_dataset import RealcamvidDataset

opt = opt_dict["wan2.1_t2v_1.3b"]
opt.load_da3_cam, opt.load_depth, opt.load_conf = True, True, True
dataset = RealcamvidDataset(opt, training=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
import torch
from src.options import opt_dict
from src.data.realcamvid_dataset import RealcamvidDataset

opt = opt_dict["wan2.1_t2v_1.3b"]
opt.load_da3_cam, opt.load_depth, opt.load_conf = True, True, True
dataset = RealcamvidDataset(opt, training=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

data = next(iter(dataloader))
for k, v in data.items():
    print(f"{k}: {v.shape if isinstance(v, torch.Tensor) else v}")

import imageio.v2 as iio
from src.utils import *

I = 0

print(data["uid"][I])
iio.mimwrite("temp.mp4", tensor_to_video(data["image"][I]), fps=30)
print(data["C2W"][0, 0], data["fxfycxcy"][0, 0])

if "depth" in data:
    depths, C2W, fxfycxcy = data["depth"], data["C2W"], data["fxfycxcy"]
    print("Depth min & max:", depths[I].min(), depths[I].max())
    iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths[I])), fps=30)

    xyzs = unproject_depth(depths[I:I+1, ...], C2W[I:I+1, ...], fxfycxcy[I:I+1, ...])[0]
    xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
    xyzs = rearrange(xyzs, "c f h w -> f c h w")
    iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

    # save_xyz_rgb_as_ply("temp.ply", xyzs, data["image"][I], ratio=0.02)
    points, colors = filter_da3_points(
        data["image"][I], data["depth"][I], data["conf"][I],
        data["C2W"][I], data["fxfycxcy"][I],
        conf_thresh_percentile=0.4,
        random_sample_ratio=0.01,
    )
    save_xyz_rgb_as_ply("temp.ply", points, colors, ratio=0.01)

if "conf" in data:
    confs = data["conf"][I]
    print("Conf min & max:", confs.min(), confs.max())
    confs = (confs - confs.min()) / (confs.max() - confs.min() + 1e-8)
    iio.mimwrite("temp_conf.mp4", tensor_to_video(confs[:, None].repeat(1, 3, 1, 1)), fps=30)


In [ ]:
import imageio.v2 as iio

render_images = render_pt3d_points(288, 512,
    points.to("cuda"), colors.to("cuda"), data["C2W"][I].to("cuda"), data["fxfycxcy"][I].to("cuda"))
iio.mimwrite("temp_render.mp4", tensor_to_video(render_images.cpu()), fps=30)

## Internal Dataset

In [ ]:
import torch
from src.options import opt_dict
from src.data.internal_dataset import InternalDataset

opt = opt_dict["wan2.1_t2v_1.3b"]
dataset = InternalDataset(opt, training=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
data = next(iter(dataloader))
for k, v in data.items():
    print(f"{k}: {v.shape if isinstance(v, torch.Tensor) else v}")

import imageio.v2 as iio
from src.utils import *

I = 0

print(data["uid"][I])
iio.mimwrite("temp.mp4", tensor_to_video(data["image"][I]), fps=30)
print(data["C2W"][0, 0], data["fxfycxcy"][0, 0])


# Visualize DA3

In [ ]:
import imageio.v2 as iio
import numpy as np
from decord import VideoReader, cpu
import torchvision.transforms as tvT
from src.options import ROOT
from src.utils import *

x = np.load(f"{ROOT}/data/RealCam-Vid-DA3/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.npz")
vr = VideoReader(str(f"{ROOT}/data/RealCam-Vid/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.mp4"), ctx=cpu(0))
v = vr[:].asnumpy()

depths, confs, W2C, intrinsics = x["depth"], x["conf"], x["extrinsics"], x["intrinsics"]
print(depths.shape, confs.shape, W2C.shape, intrinsics.shape)

v = torch.from_numpy(v).permute(0, 3, 1, 2) / 255.
v = tvT.Resize((280, 504), tvT.InterpolationMode.BICUBIC)(v).clamp(0., 1.)
print(v.shape)


In [ ]:
depths = torch.from_numpy(depths).float()
confs = torch.from_numpy(confs).float()

W2C_ = torch.eye(4).unsqueeze(0).repeat(W2C.shape[0], 1, 1)
W2C_[:, :3, :4] = torch.from_numpy(W2C).float()
C2W = inverse_c2w(W2C_)
intrinsics[:, 0, 0] /= 504
intrinsics[:, 1, 1] /= 280
intrinsics[:, 0, 2] /= 504
intrinsics[:, 1, 2] /= 280
fxfycxcy = intrinsics_to_fxfycxcy(torch.from_numpy(intrinsics).float()[None, ...])[0]

In [ ]:
iio.mimwrite("temp.mp4", tensor_to_video(v), fps=30)
iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths)), fps=30)
iio.mimwrite("temp_conf.mp4", tensor_to_video(colorize_depth(confs)), fps=30)

xyzs = unproject_depth(depths[None, ...], C2W[None, ...], fxfycxcy[None, ...])[0]
xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
xyzs = rearrange(xyzs, "c f h w -> f c h w")
iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

save_xyz_rgb_as_ply("temp.ply", xyzs, v, ratio=0.02)

# Save EMA Checkpoint

In [ ]:
import torch
from src.options import opt_dict, ROOT
from src.models import MyEMAModel
from src.models import Wan

opt = opt_dict["wan2.1_t2v_1.3b"]
model = Wan(opt)

tag, iter = "wan_t2v_cc_81x512", 15689

ema_path = f"{ROOT}/projects/VGGM/out/{tag}/checkpoints/{iter:06d}/ema_states.pth"
ema_states = MyEMAModel(model.parameters())
ema_states.load_state_dict(torch.load(ema_path, map_location="cpu", weights_only=True))
ema_states.copy_to(model.parameters())

torch.save(model.diffusion.state_dict(), f"{tag}_{iter:06d}.pth")

# Visualize DA3 for Internal Dataset

In [ ]:
from decord import VideoReader, cpu
import numpy as np
import imageio.v2 as iio
import torch
import torchvision.transforms as tvT
from src.options import DATAROOT
from src.utils import *

def build_16fps_frame_indices(num_frames: int, src_fps: float, target_fps: float = 16.):
    if num_frames <= 0:
        return np.array([], dtype=np.int64)
    if src_fps <= 0:
        return np.arange(num_frames, dtype=np.int64)
    duration = num_frames / src_fps
    target_ts = np.arange(0., duration, 1. / target_fps, dtype=np.float32)
    frame_indices = np.floor(target_ts * src_fps).astype(np.int64)
    frame_indices = np.clip(frame_indices, 0, num_frames - 1)
    return frame_indices

uid = np.random.choice(os.listdir(f"{DATAROOT}/da3"))

file_mp4 = f"{DATAROOT}/video/{uid}.mp4"
vr = VideoReader(str(file_mp4), ctx=cpu(0))

num_frames_src, fps_src = len(vr), vr.get_avg_fps()
frame_idxs_16fps = build_16fps_frame_indices(num_frames_src, fps_src, target_fps=16.)
num_frames, fps = len(frame_idxs_16fps), 16.

for i in range(1, 2):
    file = f"{DATAROOT}/da3/{uid}/{i:02d}"
    depths = np.load(f"{file}/depth.npy")
    confs = np.load(f"{file}/conf.npy")
    W2C = np.load(f"{file}/extrinsics.npy")
    intrinsics = np.load(f"{file}/intrinsics.npy")

    depths = torch.from_numpy(depths).float()
    confs = torch.from_numpy(confs).float()

    W2C_ = torch.eye(4).unsqueeze(0).repeat(W2C.shape[0], 1, 1)
    W2C_[:, :3, :4] = torch.from_numpy(W2C).float()
    C2W = inverse_c2w(W2C_)
    intrinsics[:, 0, 0] /= 504
    intrinsics[:, 1, 1] /= 280
    intrinsics[:, 0, 2] /= 504
    intrinsics[:, 1, 2] /= 280
    fxfycxcy = intrinsics_to_fxfycxcy(torch.from_numpy(intrinsics).float()[None, ...])[0]

    start_frame_idx = int(round((i - 1) * 5 * fps)) - 12 * (i - 1)
    end_frame_idx = start_frame_idx + int(round(5 * fps))
    frame_idxs = np.arange(num_frames, dtype=int)
    if start_frame_idx is not None:
        frame_idxs = frame_idxs[frame_idxs >= start_frame_idx]
    if end_frame_idx is not None:
        frame_idxs = frame_idxs[frame_idxs < end_frame_idx]

    src_frame_idxs = frame_idxs_16fps[frame_idxs]
    v = np.stack([vr[idx].asnumpy() for idx in src_frame_idxs], axis=0)

    v = torch.from_numpy(v).permute(0, 3, 1, 2) / 255.
    v = tvT.Resize((280, 504), tvT.InterpolationMode.BICUBIC)(v).clamp(0., 1.)

    print(f"[{i:02d}] {v.shape}, depth range: {depths.min()} ~ {depths.max()}, conf range: {confs.min()} ~ {confs.max()}")
    print(f"Camera pose range: {C2W[:, :3, 3].min(dim=0).values} ~ {C2W[:, :3, 3].max(dim=0).values}")
    print(f"Camera intrinsics fx range: {fxfycxcy[:, 0].min()} ~ {fxfycxcy[:, 0].max()}, fy range: {fxfycxcy[:, 1].min()} ~ {fxfycxcy[:, 1].max()}")
    print(f"Video frame indices (src): {src_frame_idxs.min()} ~ {src_frame_idxs.max()}")

    iio.mimwrite("temp.mp4", tensor_to_video(v), fps=30)
    iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths)), fps=30)
    iio.mimwrite("temp_conf.mp4", tensor_to_video(colorize_depth(confs)), fps=30)

    xyzs = unproject_depth(depths[None, ...], C2W[None, ...], fxfycxcy[None, ...])[0]
    xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
    xyzs = rearrange(xyzs, "c f h w -> f c h w")
    iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

    points, colors = filter_da3_points(
        v, depths, confs, C2W, fxfycxcy,
        conf_thresh_percentile=0.4,
        random_sample_ratio=0.1,
    )
    save_xyz_rgb_as_ply("temp.ply", points, colors)
